In [1]:
import sys                                           # 系统模块
import asyncio                                       # 异步 IO 模块
from dotenv import load_dotenv                       # 读取 .env
load_dotenv()                                        # 加载配置

if sys.platform == "win32":                          # 如果是 Windows 平台
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):  # 若当前不是 Proactor 事件循环策略
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())  # 设为 Proactor（Windows 子进程需要）

    if "ipykernel" in sys.modules:                   # 如果运行在 Jupyter 内核里
        sys.stderr = sys.__stderr__                  # 恢复标准错误输出（避免某些显示问题）

from langchain_mcp_adapters.client import MultiServerMCPClient
client=MultiServerMCPClient({
    "time":{
        "transport":"stdio",
        "command":'uvx',
        "args":[
            "mcp-server-time",
            "--local-timezone=Asia/Shanghai",
        ]
    }
})
tools=await client.get_tools()
for tool in tools:
    print(tool.name)
from langchain_core.messages import HumanMessage
import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
model=init_chat_model(
    model='qwen3.5-plus',
    model_provider='openai',
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)
agent=create_agent(
    model,
    tools
)
response=await agent.ainvoke(
    {
        "messages":HumanMessage("现在几点了")
    }
)
for message in response["messages"]:
    message.pretty_print()

C:\Users\202501942\AppData\Local\Temp\ipykernel_8472\4157907197.py:7: DeprecationWarning: 'asyncio.get_event_loop_policy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):  # 若当前不是 Proactor 事件循环策略
C:\Users\202501942\AppData\Local\Temp\ipykernel_8472\4157907197.py:7: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):  # 若当前不是 Proactor 事件循环策略


get_current_time
convert_time
get_time_ago
================================ Human Message =================================

现在几点了
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_e26fc6c5f6b245ae9e884a47)
 Call ID: call_e26fc6c5f6b245ae9e884a47
  Args:
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-11-13T14:53:00+08:00",\n  "day_of_week": "Friday",\n  "is_dst": false\n}', 'id': 'lc_969248d7-230e-454d-8120-55feeed3ba32'}]
================================== Ai Message ==================================
Tool Calls:
   (call_50997c6f66bb461dbf48053e)
 Call ID: call_50997c6f66bb461dbf48053e
  Args:
================================= Tool Message =================================
Name: 

Error:  is not a valid tool, try one of [get_current_time, convert_time, get_time_ago].
============

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient
client=MultiServerMCPClient(
    {
        "kiwi-com-flight-search":{
            "transport":"http",
            "url":'https://mcp.kiwi.com',
        }
    }
)
tools=await client.get_tools()
for tool in tools:
    print(tool.name)
model=init_chat_model(
    model="qwen3.5-plus",
    model_provider='openai',
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)
agent=create_agent(
    model,
    tools,
    system_prompt="你是一名旅行智能助手，帮用户找到合适的航班"
)
response=await agent.ainvoke({
    "messages":[HumanMessage("帮我查询下周一深圳飞北京的航班")]
})
for message in response["messages"]:
    message.pretty_print()

search-flight
feedback-to-devs
================================ Human Message =================================

帮我查询下周一深圳飞北京的航班
================================== Ai Message ==================================
Tool Calls:
  search-flight (call_167106a59bd64e36ae84a3e5)
 Call ID: call_167106a59bd64e36ae84a3e5
  Args:
    flyFrom: 深圳
    flyTo: 北京
    departureDate: 22/12/2025
================================= Tool Message =================================
Name: search-flight

[{'type': 'text', 'text': '{\n  "query": "深圳 → 北京 on 22/12/2025, 1 adult",\n  "currency": null,\n  "passengers": null,\n  "resultsCount": 0,\n  "itineraries": [],\n  "searchTimeMs": 0,\n  "error": "These dates are in the past: departureDate (22/12/2025). Today is 13/11/2026 — please use dates today or later."\n}', 'id': 'lc_8ab3de6b-0804-4d68-b474-46ca71f70e1b'}]
================================== Ai Message ==================================
Tool Calls:
  search-flight (call_1a8e86e902b549c3834703c2)
 Call ID: cal